# init_recording.ipynb — Synchronous Sensor Recording

This notebook **records**. It produces raw data only.

**Output:** `SESSION_DIR/events.jsonl` + `SESSION_DIR/video.mp4`

These files are then processed by `ground_truth_classifier.ipynb`.

**Hardware sources:**
- **Polar H10** (Bluetooth LE) — R-R intervals
- **Camera** (built-in / Continuity) — video for FER
- **ESP32 + AD8232 + SGP40** (WebSocket on port 81) — plant voltage @ 100Hz and VOC index @ 2Hz
- **CO2 (SCD41)** — placeholder, not yet connected

**Mock mode:** Polar, camera and ESP32 use real hardware. When `USE_MOCK_ESP32 = True`,
plant voltage and VOC are synthesised directly into the event store (useful when the
ESP32 is not yet on the network).


## 1  Imports & Configuration

In [1]:
import os
os.environ["OPENCV_AVFOUNDATION_SKIP_AUTH"] = "1"  # macOS camera fix
import asyncio, json, math, struct, threading, time, warnings
from pathlib import Path

import numpy as np

try:
    from bleak import BleakScanner, BleakClient
    BLEAK_OK = True
except ImportError:
    BLEAK_OK = False
    print("bleak not installed — pip install bleak")

try:
    import cv2
    CV2_OK = True
except ImportError:
    CV2_OK = False
    print("opencv not installed — pip install opencv-python")

try:
    import websockets
    WEBSOCKETS_OK = True
except ImportError:
    WEBSOCKETS_OK = False
    print("websockets not installed — pip install websockets")

# ══════════════════════════════════════════════════════════════════
# CONFIGURATION — adjust here
# ══════════════════════════════════════════════════════════════════
import datetime
PROBAND_ID    = "proband_01"            # ← change per session
_ts           = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
SESSION_ID    = f"{_ts}_{PROBAND_ID}"
SESSION_DIR   = Path("./sessions") / SESSION_ID
DURATION_S    = 660                      # recording length in seconds
BASELINE_S    = 60                       # resting phase at the start (no video)
VIDEO_FPS     = 30                       # camera frame rate

# ESP32 plant + VOC streamer (see plant_sensor_script.ino)
ESP32_IP        = "192.168.0.100"        # ← IP shown on the OLED screen
ESP32_PORT      = 81
ESP32_RATE_HZ   = 100                    # output rate the ESP32 sends at

# Mock switch — when True, plant_voltage + voc are synthesised locally
# (Polar and camera still need real hardware)
USE_MOCK_ESP32  = True

SESSION_DIR.mkdir(parents=True, exist_ok=True)
EVENT_STORE = SESSION_DIR / "events.jsonl"
VIDEO_FILE  = SESSION_DIR / "video.mp4"

print(f"Session folder : {SESSION_DIR.resolve()}")
print(f"Duration       : {DURATION_S}s  |  Baseline: {BASELINE_S}s")
print(f"ESP32          : {'MOCK' if USE_MOCK_ESP32 else f'ws://{ESP32_IP}:{ESP32_PORT}'}")


Session folder : /Users/jakobjetter/Documents/Universität zu Köln/2. Semester/COIN Seminar/Project/Local/Programming/Label Generator V03/sessions/2026-06-01_17-04_proband_01
Duration       : 660s  |  Baseline: 60s
CO2/VOC mock   : True


## 2  Event Store

All sensors write independently into one shared JSONL file. Each event has
exactly three fields: `ts` (Unix ms), `sensor`, `value`.

```
{"ts": 1700000000123, "sensor": "hr_rr",        "value": 812.3}
{"ts": 1700000000200, "sensor": "camera_frame",  "value": 42}
{"ts": 1700000001000, "sensor": "co2",           "value": 412.0}
{"ts": 1700000001000, "sensor": "voc",           "value": 95.0}
{"ts": 1700000001100, "sensor": "plant_voltage", "value": 0.823}
```

No sensor is the master clock. Resampling onto a common grid happens later in
`ground_truth_classifier.ipynb`.


In [2]:
def write_event(ts_ms, sensor, value):
    """Write one event to the store (thread-safe via append)."""
    with open(EVENT_STORE, "a") as f:
        f.write(json.dumps({"ts": int(ts_ms), "sensor": sensor,
                            "value": value}) + "\n")

# Clear the store (new session)
EVENT_STORE.write_text("")
print(f"Event store initialised: {EVENT_STORE}")


Event store initialised: sessions/2026-06-01_17-04_proband_01/events.jsonl


## 3  Sensor Functions

### 3a  Polar H10 — BLE R-R Intervals

In [3]:
HR_CHAR = "00002a37-0000-1000-8000-00805f9b34fb"

def polar_handler(sender, data):
    """BLE callback: decode R-R intervals (Bluetooth SIG 2011).
    R-R is encoded as little-endian uint16 in units of 1/1024 s."""
    flags      = data[0]
    hr_16_bit  = flags & 0x01
    rr_present = flags & 0x10
    offset     = 3 if hr_16_bit else 2
    if flags & 0x08:
        offset += 2                        # skip Energy Expended field
    if rr_present:
        while offset + 1 < len(data):
            rr_raw = struct.unpack_from("<H", data, offset)[0]
            rr_ms  = rr_raw / 1024 * 1000  # 1/1024 s → ms
            write_event(time.time() * 1000, "hr_rr", round(rr_ms, 1))
            offset += 2

async def run_polar(duration_s):
    if not BLEAK_OK:
        raise RuntimeError("bleak not installed")
    print("Scanning for Polar H10 ...")
    devices = await BleakScanner.discover()
    polar   = next((d for d in devices if d.name and "Polar" in d.name), None)
    if polar is None:
        print("No Polar H10 found"); return
    print(f"Connected: {polar.name}")
    async with BleakClient(polar.address) as client:
        await client.start_notify(HR_CHAR, polar_handler)
        await asyncio.sleep(duration_s)
        await client.stop_notify(HR_CHAR)
    print("Polar done.")


### 3b  Camera — Frame Timestamps from System Clock

In [4]:
def run_camera(duration_s):
    import time

    # Prefer index 1 (Mac camera), fall back to 0 (e.g. iPhone Continuity).
    # If only one camera exists, it becomes index 0 → fallback still works.
    selected_index = None
    for i in [1, 0]:
        cap_test = cv2.VideoCapture(i)
        if cap_test.isOpened():
            ok, frame = cap_test.read()
            cap_test.release()
            if ok and frame is not None and frame.size > 0:
                selected_index = i
                print(f"Using camera index {i}")
                break
            else:
                print(f"Camera {i} opened but returned no frames — skipping")

    if selected_index is None:
        print("No working camera found"); return

    cap = cv2.VideoCapture(selected_index)
    time.sleep(2)                       # macOS warm-up
    for _ in range(10):
        cap.read()                      # discard warm-up frames

    # Read resolution from the camera
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or fps > 120:
        fps = VIDEO_FPS
    print(f"Resolution: {w}x{h} @ {fps:.0f}fps")

    # avc1 (H.264) works reliably on macOS; fall back to mp4v
    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    writer = cv2.VideoWriter(str(VIDEO_FILE), fourcc, fps, (w, h))
    if not writer.isOpened():
        print("avc1 failed — trying mp4v")
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(str(VIDEO_FILE), fourcc, fps, (w, h))

    frame_id = 0
    t_end = time.time() + duration_s
    while time.time() < t_end:
        ok, frame = cap.read()
        if not ok or frame is None or frame.size == 0:
            time.sleep(0.01); continue
        ts_ms = time.time() * 1000       # per-frame system timestamp
        write_event(ts_ms, "camera_frame", frame_id)
        writer.write(frame)
        frame_id += 1

    cap.release()
    writer.release()
    print(f"Camera done: {frame_id} frames -> {VIDEO_FILE}")


### 3c  Environment Sensors — Plant voltage + VOC (ESP32 / WebSocket), CO2 placeholder

In [5]:
# ── CO2 placeholder (SCD41 not yet connected) ──────────────────
def read_scd41():
    """Returns (co2_ppm, temp_C, rh_%) once the SCD41 is wired in.
    Driver: adafruit_circuitpython_scd4x. Currently a placeholder."""
    raise NotImplementedError("SCD41 hardware not yet connected")

# ── ESP32 WebSocket listener (plant voltage @ 100Hz + VOC) ─────
#
# Wire protocol: every ~500ms the ESP32 broadcasts a JSON batch:
#     {"type":"data", "timestamp":<ms>, "sampleRate":100,
#      "samples":50, "voc":<index>, "voltages":[mV, mV, ...]}
#
# Per-sample timestamping: the batch arrives at host time T with N
# voltage samples covering the previous N*10ms. We back-calculate
# each sample timestamp as T - (N-1-i) * (1000/sampleRate) so that
# the last sample is stamped at T. Voltages are converted from mV
# to V to match the existing analysis pipeline.

async def esp32_listener(duration_s):
    """Connect to ESP32 over WebSocket, write voltage + VOC events
    until duration_s elapses or the recording session ends."""
    if not WEBSOCKETS_OK:
        raise RuntimeError("pip install websockets")
    import websockets
    uri = f"ws://{ESP32_IP}:{ESP32_PORT}"
    t_end = time.time() + duration_s
    n_batches = 0
    while time.time() < t_end:
        try:
            async with websockets.connect(uri, ping_interval=20, ping_timeout=10) as ws:
                print(f"ESP32 connected: {uri}")
                async for message in ws:
                    if time.time() >= t_end:
                        break
                    try:
                        data = json.loads(message)
                    except Exception:
                        continue
                    if data.get("type") != "data":
                        continue
                    voltages = data.get("voltages", [])
                    sample_rate = data.get("sampleRate", ESP32_RATE_HZ)
                    n = len(voltages)
                    if n == 0:
                        continue
                    arrival_ms = time.time() * 1000
                    step_ms    = 1000.0 / sample_rate
                    # Back-calculate per-sample timestamps
                    for i, mv in enumerate(voltages):
                        ts_ms = arrival_ms - (n - 1 - i) * step_ms
                        volts = float(mv) / 1000.0       # mV -> V
                        write_event(ts_ms, "plant_voltage", round(volts, 6))
                    # VOC stamped at batch arrival
                    voc = data.get("voc")
                    if voc is not None:
                        write_event(arrival_ms, "voc", round(float(voc), 1))
                    n_batches += 1
        except Exception as e:
            print(f"ESP32 disconnected ({e}); retrying in 2s ...")
            await asyncio.sleep(2)
    print(f"ESP32 listener done — {n_batches} batches received")

# ── Mock: synthesise plant + VOC directly (no fake WebSocket) ──
_rng_env = np.random.default_rng(99)

def run_mock_esp32(duration_s):
    """Write synthetic plant_voltage @ 100Hz + voc @ 2Hz into the event
    store. Used when USE_MOCK_ESP32 = True (e.g. ESP32 not on network)."""
    t_end = time.time() + duration_s
    next_voc = time.time()
    last_v   = 0.50
    while time.time() < t_end:
        ts_ms = time.time() * 1000
        # plant voltage: slow drift + small noise around 0.5V
        last_v += float(_rng_env.normal(0, 0.001))
        last_v  = float(np.clip(last_v, 0.30, 0.70))
        write_event(ts_ms, "plant_voltage", round(last_v, 6))
        # VOC roughly every 500ms (matches real ESP32 batch cadence)
        if time.time() >= next_voc:
            voc = 95 + float(_rng_env.normal(0, 6))
            write_event(ts_ms, "voc", round(voc, 1))
            next_voc = time.time() + 0.5
        time.sleep(0.01)        # 100Hz

def run_env_sensors(duration_s):
    """Dispatcher: real ESP32 listener or mock."""
    if USE_MOCK_ESP32:
        run_mock_esp32(duration_s)
    else:
        # Real ESP32 — async WebSocket in its own event loop
        loop = asyncio.new_event_loop()
        try:
            loop.run_until_complete(esp32_listener(duration_s))
        finally:
            loop.close()

    # CO2 placeholder: try to read once at the end to log status
    try:
        co2, _, _ = read_scd41()
        write_event(time.time() * 1000, "co2", co2)
    except NotImplementedError:
        pass  # not yet wired in


## 4  Start Recording

All sensors start at the same time. The system clock (`time.time()`) is the
common time base — no sensor is the master.

**Flow:**
1. Event store already initialised (Block 2)
2. Camera + environment sensors start in their own threads
3. Polar H10 runs in the asyncio loop (BLE requires async)
4. After `DURATION_S` seconds everything stops automatically
5. Short status report

Afterwards: open `events.jsonl` and `video.mp4` in `SESSION_DIR` →
run `ground_truth_classifier.ipynb`.


In [6]:
if not BLEAK_OK:
    raise RuntimeError("pip install bleak  (Polar H10 requires bleak)")
if not CV2_OK:
    raise RuntimeError("pip install opencv-python  (camera requires opencv)")
if not USE_MOCK_ESP32 and not WEBSOCKETS_OK:
    raise RuntimeError("pip install websockets  (real ESP32 needs the websockets client)")

T0_MS = int(time.time() * 1000)
# Save session metadata (important for later synchronisation)
meta = {"session_id": SESSION_ID, "proband_id": PROBAND_ID,
        "t0_ms": T0_MS, "duration_s": DURATION_S, "baseline_s": BASELINE_S,
        "esp32_uri": f"ws://{ESP32_IP}:{ESP32_PORT}" if not USE_MOCK_ESP32 else "mock",
        "recorded_at": datetime.datetime.now().isoformat()}
(SESSION_DIR / "session_meta.json").write_text(json.dumps(meta, indent=2))
print(f"Session start : {T0_MS} ms  ({time.strftime('%H:%M:%S')})")
print(f"Baseline      : first {BASELINE_S}s rest — please sit still, no video")
print(f"Total length  : {DURATION_S}s")
print(f"ESP32         : {'MOCK' if USE_MOCK_ESP32 else f'ws://{ESP32_IP}:{ESP32_PORT}'}")
print()

errors = []

def _run_camera():
    try: run_camera(DURATION_S)
    except Exception as e: errors.append(f"Camera: {e}")

def _run_env():
    try: run_env_sensors(DURATION_S)
    except Exception as e: errors.append(f"Env (ESP32): {e}")

# Start threads
t_cam = threading.Thread(target=_run_camera, daemon=True)
t_env = threading.Thread(target=_run_env,    daemon=True)
t_cam.start()
t_env.start()

# Polar in the asyncio loop (Jupyter: await usable directly)
await run_polar(DURATION_S)

# Wait for threads to finish
t_cam.join()
t_env.join()

if errors:
    print("\nErrors during recording:")
    for e in errors: print(f"  {e}")
else:
    print("\nRecording complete — no errors")

# Status report
from collections import Counter
events = [json.loads(l) for l in EVENT_STORE.read_text().strip().split("\n") if l.strip()]
by_sensor = Counter(e["sensor"] for e in events)
print(f"\nEvent store: {len(events)} events in {EVENT_STORE.name}")
for s, n in sorted(by_sensor.items()):
    print(f"  {s:<20} {n:>6}")
print(f"\nNext step: run ground_truth_classifier.ipynb")
print(f"SESSION_DIR = '{SESSION_DIR.resolve()}'")


Session start : 1780326248484 ms  (17:04:08)
Baseline      : first 60s rest — please sit still, no video
Total length  : 660s
CO2/VOC       : MOCK

Scanning for Polar H10 ...
Using camera index 1
Resolution: 1280x720 @ 30fps
Connected: Polar H10 E827092E
Camera done: 19809 frames -> sessions/2026-06-01_17-04_proband_01/video.mp4
Polar done.

Recording complete — no errors

Event store: 21826 events in events.jsonl
  camera_frame          19809
  co2                     658
  hr_rr                   701
  voc                     658

Next step: run ground_truth_classifier.ipynb
SESSION_DIR = '/Users/jakobjetter/Documents/Universität zu Köln/2. Semester/COIN Seminar/Project/Local/Programming/Label Generator V03/sessions/2026-06-01_17-04_proband_01'
